In [1]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [2]:
import os, glob, json
import numpy as np
import pandas as pd
import pyarrow.parquet as pq

BASE_DIR = "/content/drive/MyDrive/Mining of massive dataset/preprocessing_output"
LOOKUP_PATH = os.path.join(BASE_DIR, "taxi_zone_lookup_grid.csv")

In [3]:
def get_month_paths(dataset: str, year: int, month: int):
    mm = f"{month:02d}"
    year_dir = os.path.join(BASE_DIR, f"{dataset}_data", f"{dataset}_{year}")
    vol_path = os.path.join(year_dir, f"{mm}_volume.csv")
    flow_path = os.path.join(year_dir, f"{mm}_flow.parquet")
    return vol_path, flow_path

GRID_H, GRID_W = 10, 20
TIME_INTERVAL = "1H"
YEAR = 2018
MONTH = 1
DATASET = "yellow"

In [4]:
VOL_PATH, FLOW_PATH = get_month_paths(DATASET, YEAR, MONTH)
print(VOL_PATH, os.path.exists(VOL_PATH))
print(FLOW_PATH, os.path.exists(FLOW_PATH))

/content/drive/MyDrive/Mining of massive dataset/preprocessing_output/yellow_data/yellow_2018/01_volume.csv True
/content/drive/MyDrive/Mining of massive dataset/preprocessing_output/yellow_data/yellow_2018/01_flow.parquet True


### tạo map LocationID -> (i,j)

In [5]:
grid = pd.read_csv(LOOKUP_PATH)
grid["Grid_X"] = grid["Grid_X"].astype(int)
grid["Grid_Y"] = grid["Grid_Y"].astype(int)

loc2ij = dict(zip(grid["LocationID"], zip(grid["Grid_X"], grid["Grid_Y"])))

print("Grid_X range:", grid["Grid_X"].min(), grid["Grid_X"].max())
print("Grid_Y range:", grid["Grid_Y"].min(), grid["Grid_Y"].max())

Grid_X range: 0 9
Grid_Y range: 0 19


### time_map từ volume

In [6]:
vol_df = pd.read_csv(VOL_PATH)
vol_df["time_bin"] = pd.to_datetime(vol_df["time_bin"], errors="coerce")
vol_df = vol_df.dropna(subset=["time_bin"])

vol_df["slot"] = vol_df["time_bin"].dt.floor(TIME_INTERVAL)

time_index = sorted(vol_df["slot"].unique())
time_map = {t:i for i,t in enumerate(time_index)}
T = len(time_index)

print("Total time slots (1H):", T)

Total time slots (1H): 791


/tmp/ipykernel_21498/1986080695.py:6: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  vol_df["slot"] = vol_df["time_bin"].dt.floor(TIME_INTERVAL)


### khởi tạo volumn + flow

In [7]:
volume = np.zeros((T, GRID_H, GRID_W, 2), dtype=np.float32)

flow = np.zeros((2, T, GRID_H, GRID_W, GRID_H, GRID_W), dtype=np.float32)

print("volume shape:", volume.shape)
print("flow shape:", flow.shape)

volume shape: (791, 10, 20, 2)
flow shape: (2, 791, 10, 20, 10, 20)


### group thành 1H

In [8]:
vol_df = vol_df.rename(columns={"locationid": "LocationID"})
vol_df["LocationID"] = pd.to_numeric(vol_df["LocationID"], errors="coerce").astype("Int64")
vol_df = vol_df.dropna(subset=["LocationID"])
vol_df["LocationID"] = vol_df["LocationID"].astype(int)

vol_df["start_volume"] = pd.to_numeric(vol_df["start_volume"], errors="coerce").fillna(0).astype(np.float32)
vol_df["end_volume"]   = pd.to_numeric(vol_df["end_volume"], errors="coerce").fillna(0).astype(np.float32)

vol_hour = (vol_df.groupby(["slot","LocationID"], as_index=False)[["start_volume","end_volume"]]
            .sum())

miss_map = 0
for r in vol_hour.itertuples(index=False):
    t = r.slot
    loc = int(r.LocationID)
    if t not in time_map:
        continue
    if loc not in loc2ij:
        miss_map += 1
        continue
    ti = time_map[t]
    i, j = loc2ij[loc]
    volume[ti, i, j, 0] += float(r.start_volume)
    volume[ti, i, j, 1] += float(r.end_volume)

print("Done volume. missing map:", miss_map)
print("volume nonzero:", np.count_nonzero(volume))

Done volume. missing map: 0
volume nonzero: 80068


### Fill flow từ flow.parquet

In [9]:
pf = pq.ParquetFile(FLOW_PATH)

flow_df = pf.read(columns=["time_bin","pulocationid","dolocationid","flow_count"]).to_pandas()

flow_df["time_bin"] = pd.to_datetime(flow_df["time_bin"], errors="coerce")
flow_df = flow_df.dropna(subset=["time_bin"])
flow_df["slot"] = flow_df["time_bin"].dt.floor(TIME_INTERVAL)

flow_df["pulocationid"] = pd.to_numeric(flow_df["pulocationid"], errors="coerce").astype("Int64")
flow_df["dolocationid"] = pd.to_numeric(flow_df["dolocationid"], errors="coerce").astype("Int64")
flow_df = flow_df.dropna(subset=["pulocationid","dolocationid"])
flow_df["pulocationid"] = flow_df["pulocationid"].astype(int)
flow_df["dolocationid"] = flow_df["dolocationid"].astype(int)
flow_df["flow_count"] = pd.to_numeric(flow_df["flow_count"], errors="coerce").fillna(0).astype(np.float32)

flow_hour = (flow_df.groupby(["slot","pulocationid","dolocationid"], as_index=False)["flow_count"]
             .sum())

miss_map = 0
for r in flow_hour.itertuples(index=False):
    t = r.slot
    pu = int(r.pulocationid)
    do = int(r.dolocationid)
    c  = float(r.flow_count)

    if t not in time_map:
        continue
    if pu not in loc2ij or do not in loc2ij:
        miss_map += 1
        continue

    ti = time_map[t]
    i, j = loc2ij[pu]
    k, l = loc2ij[do]

    flow[0, ti, i, j, k, l] += c
    if ti > 0:
        flow[1, ti-1, i, j, k, l] += c

print("Done flow. missing map:", miss_map)
print("flow nonzero:", np.count_nonzero(flow))

/tmp/ipykernel_21498/1503306904.py:9: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  flow_df["slot"] = flow_df["time_bin"].dt.floor(TIME_INTERVAL)


Done flow. missing map: 0
flow nonzero: 660711


### Normalize

In [10]:
split = int(T * 0.8)

volume_train = volume[:split]
flow_train = flow[:, :split]

volume_max = volume_train.max()
flow_max = flow_train.max()

volume = volume / (volume_max + 1e-6)
flow = flow / (flow_max + 1e-6)

print("volume_max:", volume_max, "flow_max:", flow_max)
print("Normalized.")

volume_max: 6113.0 flow_max: 2271.0
Normalized.


### Split & save npz

In [14]:
OUT_DIR = f"/content/drive/MyDrive/Mining of massive dataset/preprocessing_output/yellow_data/yellow_2018/"
os.makedirs(OUT_DIR, exist_ok=True)

split = int(T * 0.8)

volume_train = volume[:split]
volume_test  = volume[split:]

flow_train = flow[:, :split]
flow_test  = flow[:, split:]

np.savez(os.path.join(OUT_DIR, "volume_train.npz"), volume=volume_train)
np.savez(os.path.join(OUT_DIR, "volume_test.npz"), volume=volume_test)
np.savez(os.path.join(OUT_DIR, "flow_train.npz"), flow=flow_train)
np.savez(os.path.join(OUT_DIR, "flow_test.npz"), flow=flow_test)

print("Saved to:", OUT_DIR)
print("Volume shape:", volume.shape)
print("Flow shape:", flow.shape)

Saved to: /content/drive/MyDrive/Mining of massive dataset/preprocessing_output/yellow_data/yellow_2018/
Volume shape: (791, 10, 20, 2)
Flow shape: (2, 791, 10, 20, 10, 20)


### tạo data.json

In [15]:
config = {
  "volume_train": os.path.join(OUT_DIR, "volume_train.npz"),
  "volume_test": os.path.join(OUT_DIR, "volume_test.npz"),
  "flow_train": os.path.join(OUT_DIR, "flow_train.npz"),
  "flow_test": os.path.join(OUT_DIR, "flow_test.npz"),
  "volume_train_max": 1.0,
  "flow_train_max": 1.0,
  "timeslot_sec": 3600,
  "threshold": 0
}

json_path = os.path.join(OUT_DIR, "data.json")
with open(json_path, "w") as f:
    json.dump(config, f)

print("Created:", json_path)

Created: /content/drive/MyDrive/Mining of massive dataset/preprocessing_output/yellow_data/yellow_2018/data.json


In [13]:
import json

with open(json_path) as f:
    cfg = json.load(f)

print("volume keys:", np.load(cfg["volume_train"]).files)
print("flow keys:", np.load(cfg["flow_train"]).files)
print("volume shape:", np.load(cfg["volume_train"])["volume"].shape)
print("flow shape:", np.load(cfg["flow_train"])["flow"].shape)

volume keys: ['volume']
flow keys: ['flow']
volume shape: (632, 10, 20, 2)
flow shape: (2, 632, 10, 20, 10, 20)
